# Lecture 6.2: Capturing & Resuming Session IDs

**Starter notebook (code along).** Pre-built cells are complete. The cells marked **[LIVE]** are empty, so you can type them along with the video.

A plain `query()` call starts a brand new session. If you send a second `query()` call, the agent has no memory of the first one. In this notebook you will fix that. You will pick a session back up, three different ways, and see the agent remember something that is **not written in any file**.

## The five ideas

| # | Idea | What you use |
| :-: | :-- | :-- |
| 1 | **Capture the session ID** | `ResultMessage.session_id`, kept in a variable |
| 2 | **Resume by ID** | `ClaudeAgentOptions(resume=<session_id>)` |
| 3 | **Continue the most recent session** | `ClaudeAgentOptions(continue_conversation=True)` |
| 4 | **Choose between resume and continue** | Continue needs no ID. Resume needs the ID you kept. |
| 5 | **Multi-turn without managing IDs** | `ClaudeSDKClient` tracks the session for you |

## Cell map

Every code cell in this notebook has a number in its heading. The cell numbers match the lecture, so you can always find your place.

| Cell | What it does |
| :-: | :-- |
| 1 | Install the SDK |
| 2 | Load the API key from Colab Secrets |
| 3 | Choose the model |
| 4 | Create the Seed Library files and record their hashes |
| 5 | Verify the files before any agent runs |
| 6 | Define the prompts |
| 7 | First call: capture the session ID |
| 8 | Resume by ID |
| 9 | Ground truth: what actually happened |
| 10 | Continue the most recent session |
| 11 | Resume versus continue |
| 12 | `ClaudeSDKClient`: two turns, no ID handling |
| 13 | Final verification |

## A note on `await`

Colab notebooks allow **top-level `await`**. That is why the cells that run an agent start directly with `async for` or `async with`. You do not wrap anything in `asyncio.run()`.

## Cell 1 [PRE-BUILT]: Install the Claude Agent SDK

This cell installs `claude-agent-sdk`, the Python package that lets your code run Claude's agent loop. The package bundles the Claude Code runtime it drives, so this is the only install the notebook needs.

| Part | Meaning |
| :-- | :-- |
| `!pip install` | The `!` runs a shell command from a notebook cell. |
| `claude-agent-sdk==0.2.167` | The exact version, pinned. Every example in this notebook was tested against it. |
| `-q` | Quiet mode, so the output stays short. |

The comment in the cell explains the pin. Pinning gives you the same behaviour as the video. You can install the latest version or any version you prefer, but newer versions may behave differently from what you see on screen.

If the package is already present at this version in your current session, `pip` notices and finishes almost immediately. Run the cell and wait for it to finish before moving on.

In [ ]:
# Cell 1: Install the Claude Agent SDK
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

## Cell 2 [PRE-BUILT]: Load your API key from Colab Secrets

The agent calls Claude on your behalf, so it needs your Anthropic API key. Colab Secrets keeps the key out of your notebook, so you never paste it into a cell.

**Create the key (once):** sign in to the Claude Console at [platform.claude.com](https://platform.claude.com/), open **API Keys**, and create a key. Copy it.

**Add it to Colab:**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on.

The cell reads the secret with `userdata.get("ANTHROPIC_API_KEY")` and writes it into `os.environ`. The SDK looks for the key in that environment variable.

*Running locally instead of in Colab?* Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Python.

In [ ]:
# Cell 2: Load the API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3 [PRE-BUILT]: Choose the model

This cell sets one variable, `MODEL_NAME`. Every agent call in this notebook passes `model=MODEL_NAME`, so **changing this one line changes the model for the entire notebook**.

The default is `claude-haiku-4-5`, the Haiku model. It is fast and inexpensive, which suits a notebook where you run many small agent calls. To see which models are currently available, follow the link in the comment. Model names change with new releases, so that page is the source of truth.

In [ ]:
# Cell 3: Choose the model for the whole notebook
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4 [PRE-BUILT]: Create the Seed Library files

The demo scenario is a small community **Seed Library**. This cell creates two files under `/content/seedlibrary_demo/`.

| File | Contents |
| :-- | :-- |
| `catalog.txt` | Six seed varieties with packet counts. One count is clearly the largest. |
| `notes.txt` | A few short lines of swap-day logistics. |

### The rule that makes this demo honest

Two facts will appear **only in the prompts** you send: a favourite seed, and a swap-day table number. Neither fact is written to disk. If the agent can answer questions about them later, the only possible source is the **session**, not your files.

### Hashes as proof

The cell also records a **SHA-256 hash** of each file. A hash is a fingerprint of a file's exact contents. If a file changes by even one character, its hash changes. We compare hashes at the end to prove the agent never modified your files. The helper `sha256_of()` is reused in later cells.

Run the cell. It prints the hash of `catalog.txt` so you can see what a fingerprint looks like.

In [ ]:
# Cell 4: Create the Seed Library files and record their SHA-256 hashes
import hashlib
from pathlib import Path

LIBRARY_DIR = Path("/content/seedlibrary_demo")
LIBRARY_DIR.mkdir(parents=True, exist_ok=True)

CATALOG_PATH = LIBRARY_DIR / "catalog.txt"
NOTES_PATH = LIBRARY_DIR / "notes.txt"

catalog_lines = [
    "marigold: 40 packets",
    "basil: 12 packets",
    "okra: 75 packets",
    "zinnia: 28 packets",
    "cosmos: 33 packets",
    "tomato: 21 packets",
]

notes_lines = [
    "Swap day starts at 10:00 AM in the community hall.",
    "Volunteers arrive at 9:00 AM to set up the seed shelves.",
    "Bring labeled envelopes if you want to trade your own seeds.",
]

CATALOG_PATH.write_text("\n".join(catalog_lines) + "\n")
NOTES_PATH.write_text("\n".join(notes_lines) + "\n")


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


CATALOG_HASH = sha256_of(CATALOG_PATH)
NOTES_HASH = sha256_of(NOTES_PATH)

print("catalog.txt SHA-256:", CATALOG_HASH)

## Cell 5 [PRE-BUILT]: Verify the tree and contents

Before any agent runs, confirm exactly what is on disk. This cell prints the folder tree, then the full contents of each file.

It also scans both files for the words `favourite`, `favorite` and `table`. We are about to tell the agent a favourite seed and a table number. This check proves those facts are **not** hiding in the files.

**What to look for:**

- Two files, `catalog.txt` and `notes.txt`.
- `okra` has the largest packet count.
- The scan reports `none` for both files.

In [ ]:
# Cell 5: Verify the tree and contents
print("Folder tree:")
print(LIBRARY_DIR)
for path in sorted(LIBRARY_DIR.rglob("*")):
    print("  ", path.relative_to(LIBRARY_DIR))

print()
for path in sorted(LIBRARY_DIR.glob("*.txt")):
    print(f"--- {path.name} ---")
    print(path.read_text())

print("Scan for facts that must only live in prompts:")
for path in sorted(LIBRARY_DIR.glob("*.txt")):
    text = path.read_text().lower()
    hits = [word for word in ("favourite", "favorite", "table") if word in text]
    print(f"  {path.name}: found -> {hits or 'none'}")

## Cell 6 [PRE-BUILT]: Define the prompts

Prompts are long strings, so we define them once as variables and reuse them. That keeps the agent-running cells short and easy to read.

| Variable | Used in | Purpose |
| :-- | :-- | :-- |
| `FIRST_PROMPT` | Cell 7 | Asks the agent to read the catalog, **and** tells it your favourite seed. |
| `FOLLOW_UP` | Cell 8 | Asks which seed you said was your favourite. The answer is **not** in any file. |
| `CONTINUE_PROMPT` | Cell 10 | A new follow-up that needs facts from both earlier turns. |
| `CLIENT_TURN_1` | Cell 12 | First question for the client: read the notes and note a table number. |
| `CLIENT_TURN_2` | Cell 12 | Asks for the table number you mentioned. |

**One detail to notice:** the agent runs with `cwd="/content"`, but the files live in a subfolder. So `FIRST_PROMPT` and `CLIENT_TURN_1` include the full file path through an f-string. This saves the agent a search step. It is a goal prompt, not orchestration. We still let the agent decide how to do the work.

In [ ]:
# Cell 6: Define the prompts
FIRST_PROMPT = (
    f"Read {CATALOG_PATH} and tell me which seed has the most packets "
    "and how many. Also, my favourite seed is basil."
)

FOLLOW_UP = "Which seed did I say was my favourite?"

CONTINUE_PROMPT = (
    "In one sentence, remind me which seed has the most packets "
    "and which one is my favourite."
)

CLIENT_TURN_1 = (
    f"Read {NOTES_PATH} and tell me what time swap day starts. "
    "Also, the swap table number is 7."
)

CLIENT_TURN_2 = "What table number did I mention?"

## Cell 7 [LIVE]: First call, capturing the session ID

This is the first of three ways to run an agent in this notebook. Start with a mental model you should never gloss over:

> `query()` does **not** return a value. It returns an **async stream of messages** while the agent loop runs on its own. The final answer arrives on the **`ResultMessage`**, which is the last message in the stream.

Every `ResultMessage` carries a `session_id`, on success **and** on error. That ID is the handle for this conversation. We read it and keep it in a variable named `session_id`.

| Piece | Meaning |
| :-- | :-- |
| `prompt=FIRST_PROMPT` | What we ask. It includes the favourite-seed fact. |
| `allowed_tools=["Read"]` | Least privilege. The agent only needs to read one file. |
| `model=MODEL_NAME` | The model chosen in Cell 3. |
| `cwd="/content"` | The agent's working directory. This also decides **where sessions are stored and found**. |
| `isinstance(message, ResultMessage)` | Picks out the final message from the stream. |
| `message.session_id` | The ID of the session this call created. |

The SDK writes the session to disk automatically, so there is nothing for you to save. What you keep is only the ID.

**What to look for:** the agent should say that `okra` has the most packets, with a count of 75. Live agent output varies in wording from run to run. That is normal for real agents, so watch for the facts, not the exact sentences. The last line prints your session ID.

In [ ]:
# LIVE CODE - type during recording

## Cell 8 [LIVE]: Resume by ID

Now the payoff. This call sends only `FOLLOW_UP`, which asks about your favourite seed. We do **not** repeat the first prompt. We do not mention basil. The only thing connecting this call to the earlier one is a single option:

```python
resume=session_id
```

Passing a session ID to `resume` returns to that specific session. The agent loads the saved conversation, so it has the earlier context: what you asked, what it read, what you told it.

| Piece | Meaning |
| :-- | :-- |
| `prompt=FOLLOW_UP` | A question whose answer exists only in the first prompt. |
| `resume=session_id` | The ID captured in Cell 7. |
| Everything else | Unchanged from Cell 7. |

**What to look for:** the answer should be **basil**. Wording will vary between runs, so look for the word, not the sentence.

**Where does that memory come from?** Not from your files. Cell 5 proved that no file mentions a favourite seed. The memory comes from the **session being loaded again**. Resuming gives the agent its earlier conversation back. That is the idea to carry forward.

*Run Cell 7 first. If `session_id` is empty, there is nothing to resume.*

In [ ]:
# LIVE CODE - type during recording

## Cell 9 [PRE-BUILT]: Ground truth, what actually happened

Do not rely on how the answer sounded. This cell checks the facts.

| Check | Why it matters |
| :-- | :-- |
| Both session IDs | Shows the ID from Cell 7 and the ID reported by the resumed call in Cell 8. |
| Same ID or not | We **report** whether the resumed result's `session_id` equals the original. We do not assume the answer. Read what your run printed. |
| Session file count | `list_sessions(directory="/content")` lists the sessions stored for this working directory. It is a normal synchronous function, so you call it **without** `await`. |
| `catalog.txt` hash | Compares today's fingerprint with the one from Cell 4. If it matches, the agent did not touch the file. |

Your session count may be higher than you expect if you ran other agents in `/content` earlier in this runtime. That is fine. The count lists what is stored.

Sessions persist the **conversation**, not the filesystem. The file hash staying the same shows that the agent's memory of basil did not come from anything it changed on disk.

In [ ]:
# Cell 9: Ground truth
from claude_agent_sdk import list_sessions

print("Original session ID: ", session_id)
print("Resumed call reports:", resumed_session_id)
print("Same ID as original: ", resumed_session_id == session_id)

print()
sessions = list_sessions(directory="/content")
print("Session files found in /content:", len(sessions))
for session in sessions:
    print(" ", session.session_id, "|", session.summary[:60])

print()
print("catalog.txt unchanged:", sha256_of(CATALOG_PATH) == CATALOG_HASH)

## Cell 10 [LIVE]: Continue the most recent session

Resume needs an ID. **Continue** does not. Setting `continue_conversation=True` tells the SDK to pick up the most recent session in the current working directory.

| Piece | Meaning |
| :-- | :-- |
| `continue_conversation=True` | Find the most recent session for this `cwd` and add to it. |
| `prompt=CONTINUE_PROMPT` | A new follow-up that needs facts from **both** earlier turns: the largest stock and your favourite. |
| No `resume` | We never pass an ID here. |

Notice that there is no `session_id` variable in this call. You track nothing.

**What to look for:** if continue found the session from Cells 7 and 8, the one-sentence answer should mention **okra** (the largest stock) and **basil** (your favourite). The cell prints the session ID that came back, so you can compare it with Cell 9.

This notebook never sets `resume` and `continue_conversation` in the same call. Use one or the other.

In [ ]:
# LIVE CODE - type during recording

## Cell 11 [PRE-BUILT]: Resume or continue?

You have now used both options. They do the same job, returning to an earlier conversation, but they **find** the session differently.

| | `continue_conversation=True` | `resume=<session_id>` |
| :-- | :-- | :-- |
| **How the session is found** | Most recent session in the current working directory | The exact session whose ID you pass |
| **What you keep** | Nothing | The ID, in a variable, a file, or a database |
| **Several sessions at once** | Can only reach the latest one | Reaches any of them |
| **An older session** | Not possible | Possible |
| **Good fit** | One conversation at a time | Many conversations, for example one per user, or going back to an older one |

**A simple rule:** continue is simple but finds only the latest session. Resume asks you to keep the ID, and you need it whenever there is more than one session or you want an older one.

The code below is deliberately tiny. It prints what each option needed from you.

In [ ]:
# Cell 11: What each option needed from us
needed = {
    "resume": f"an ID (we kept {session_id})",
    "continue_conversation": "nothing",
}

for option, requirement in needed.items():
    print(f"{option:<22} needed -> {requirement}")

## Cell 12 [LIVE]: `ClaudeSDKClient`, two turns with no ID handling

So far, you managed the session yourself. For a conversation inside one running notebook, there is a simpler way: **`ClaudeSDKClient`**. The client tracks the session for you, so each new question automatically continues the same conversation.

| Piece | Meaning |
| :-- | :-- |
| `async with ClaudeSDKClient(options=...) as client` | Opens a connection and closes it for you when the block ends. |
| `await client.query("...")` | Sends a prompt. It returns nothing by itself. |
| `client.receive_response()` | Yields the messages for **that** prompt, up to and including its `ResultMessage`. |
| A second `client.query()` | Continues the **same** session. No `resume`. No ID. |

Run one `receive_response()` loop per `query()` call. Let each loop finish naturally. Do not `break` out of it early, because that can cause cleanup problems.

**What to look for:**

- Turn 1 should report the swap-day start time from `notes.txt`, which is 10:00 AM.
- Turn 2 should answer with the table number you mentioned, which is **7**. That number is in no file.
- The last lines print the `session_id` from each turn. Compare them and read what your run shows.

*Using top-level `async with` in a Colab cell is the same idea as top-level `async for`. No `asyncio.run()` is needed.*

In [ ]:
# LIVE CODE - type during recording

CLIENT_TURN_1 = (
    f"Read {NOTES_PATH} and tell me what time swap day starts. "
    "Also, the swap table number is 7."
)

CLIENT_TURN_2 = "What table number did I mention?"

## Cell 13 [PRE-BUILT]: Final verification

One last ground-truth check, now that every agent run is finished.

| Check | What it tells you |
| :-- | :-- |
| Re-listed sessions | The sessions stored for `/content` after all four runs. |
| Captured IDs missing from the listing | Any ID we saw during the notebook that does not appear in the listing. `none` means every ID we captured is stored. |
| `catalog.txt` and `notes.txt` hashes | Both files match their fingerprints from Cell 4. The agent only had the `Read` tool, and the hashes confirm it changed nothing. |

Together these show that everything the agent "remembered" came from loaded sessions, not from edits to your files.

In [ ]:
# Cell 13: Final verification
sessions = list_sessions(directory="/content")
listed_ids = {session.session_id for session in sessions}

captured_ids = {
    captured
    for captured in [session_id, resumed_session_id, continued_session_id, *client_session_ids]
    if captured
}

print("Session files found in /content:", len(sessions))
print("Distinct IDs captured:", len(captured_ids))
print("Captured IDs missing from the listing:", captured_ids - listed_ids or "none")

print()
print("catalog.txt unchanged:", sha256_of(CATALOG_PATH) == CATALOG_HASH)
print("notes.txt unchanged:  ", sha256_of(NOTES_PATH) == NOTES_HASH)

## Recap

1. **Capture the session ID.** Read it from `ResultMessage.session_id` and keep it in a variable.
2. **Resume by ID.** `resume=session_id` brings the agent back into that conversation with its full earlier context.
3. **Continue the most recent session.** `continue_conversation=True` needs no ID and finds the latest session in the working directory.
4. **Choose between them.** Continue is simple but finds only the latest session. Resume needs the ID, and you need it for several sessions or an older one.
5. **Skip ID handling for live conversations.** `ClaudeSDKClient` tracks the session, so each `query()` continues the same conversation.

The agent's memory of your favourite seed and your table number never touched a file. It came from the **session being loaded again**.